# Infinite DOM — Training Notebook

**Runtime:** Colab T4 (smoke test) or A100 (full training)

## Pipeline
1. Install dependencies
2. Load oracle training data
3. Baseline eval — random vs oracle
4. SFT warmup on oracle trajectories (Unsloth + LoRA)
5. GRPO RL with task curriculum (TRL)
6. Save model + eval + plots

## Hackathon Guide Alignment
- §3: SFT first, then RL (warm start)
- §6: Start with easiest task, curriculum up
- §7: Multiple reward components (progression, step, invalid, completion, thrash)
- §11: GRPO for verifiable tasks
- §16: Save LoRA adapters correctly, do not upcast 4-bit to 16-bit naively

In [ ]:
# Cell 1 — Install dependencies
!pip install -q unsloth
!pip install -q "trl>=0.12.0" transformers accelerate peft
!pip install -q httpx pydantic openai datasets matplotlib
!pip install -q "openenv-core @ git+https://github.com/meta-pytorch/OpenEnv.git"

import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB" if torch.cuda.is_available() else "")

In [ ]:
# Cell 2 — Load oracle training data
# Option A: From local file (if running locally alongside the environment)
# Option B: From HuggingFace dataset (if uploaded)

import json
from pathlib import Path

# --- Option A: Local file ---
LOCAL_PATH = Path("training/data/oracle_trajectories.jsonl")

# --- Option B: HuggingFace Hub ---
# from huggingface_hub import hf_hub_download
# LOCAL_PATH = hf_hub_download(
#     repo_id="YOUR_USERNAME/infinite-dom-data",
#     filename="oracle_trajectories.jsonl",
#     repo_type="dataset",
# )

with open(LOCAL_PATH) as f:
    records = [json.loads(line) for line in f]

print(f"Loaded {len(records)} oracle records")
print(f"Tasks represented: {sorted(set(r['task_id'] for r in records))}")
print(f"Sample record keys: {list(records[0].keys())}")
print(f"Sample instruction: {records[0]['instruction'][:80]}...")

In [ ]:
# Cell 3 — Prepare SFT dataset
# Format: system prompt + (observation, instruction) -> action JSON

from datasets import Dataset

SYSTEM_PROMPT = """You are a web agent navigating an interactive web application.
You observe an accessibility tree and must complete a booking task.

Respond with ONLY a JSON object (no markdown, no explanation):
{"action_type": "click"|"type"|"scroll"|"wait", "element_ref": "btn_1"|"inp_1"|"", "text_value": "text"|"", "scroll_delta": 0}

Rules:
- Use "type" to fill text fields (element_ref + text_value required)
- Use "click" to press buttons (element_ref required)
- Use "scroll" to scroll (scroll_delta: positive=down, negative=up)
- Use "wait" when unsure"""

def format_for_sft(record):
    user = f"Task: {record['instruction']}\n\nAccessibility Tree:\n{record['observation'][:2000]}"
    action = json.dumps(record['action'], separators=(',', ':'))
    return {
        "messages": [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user},
            {"role": "assistant", "content": action},
        ]
    }

sft_data = [format_for_sft(r) for r in records]
dataset = Dataset.from_list(sft_data)
train_test = dataset.train_test_split(test_size=0.1, seed=42)
train_ds = train_test["train"]
eval_ds = train_test["test"]

print(f"Train: {len(train_ds)} | Eval: {len(eval_ds)}")
print(f"Sample assistant output: {sft_data[0]['messages'][2]['content'][:100]}")

In [ ]:
# Cell 4 — Load base model with Unsloth (4-bit QLoRA)
# Guide §10: TRL + Unsloth for efficiency
# Guide §16: Use QLoRA, save adapters properly

from unsloth import FastLanguageModel

MODEL_ID = "unsloth/Qwen2.5-3B-Instruct"  # Small enough for T4, capable enough for tasks
MAX_SEQ_LEN = 4096

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,
    max_seq_length=MAX_SEQ_LEN,
    dtype=None,        # Auto-detect (bf16 on A100, fp16 on T4)
    load_in_4bit=True, # QLoRA — fits in T4 16GB
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,              # LoRA rank
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                     "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Memory efficient
)

print(f"Model loaded: {MODEL_ID}")
print(f"Trainable params: {model.print_trainable_parameters()}")

In [ ]:
# Cell 5 — SFT warmup training
# Guide §3: SFT first to give the model a warm start before RL

from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir="./sft_output",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    warmup_steps=50,
    logging_steps=10,
    save_steps=200,
    eval_strategy="steps",
    eval_steps=100,
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    max_seq_length=MAX_SEQ_LEN,
    seed=42,
    report_to="none",  # Change to "wandb" if WANDB_API_KEY set
)

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    args=sft_config,
)

print("Starting SFT training...")
sft_results = trainer.train()
print(f"SFT complete. Final loss: {sft_results.training_loss:.4f}")

In [ ]:
# Cell 6 — Save SFT model (CORRECTLY — Guide §16)
# WARNING: Do NOT upcast 4-bit to 16-bit and merge naively.
# Save LoRA adapters separately, or use Unsloth's proper merge path.

# Option A: Save LoRA adapters only (recommended, safe)
model.save_pretrained("./sft_lora_adapters")
tokenizer.save_pretrained("./sft_lora_adapters")
print("Saved LoRA adapters to ./sft_lora_adapters")

# Option B: Merge and save full model (use Unsloth's method)
# model.save_pretrained_merged("./sft_merged", tokenizer, save_method="merged_16bit")
# print("Saved merged model to ./sft_merged")

# Option C: Push to HuggingFace Hub
# model.push_to_hub("YOUR_USERNAME/infinite-dom-sft", token=os.environ.get("HF_TOKEN"))
# tokenizer.push_to_hub("YOUR_USERNAME/infinite-dom-sft", token=os.environ.get("HF_TOKEN"))
# print("Pushed to HuggingFace Hub")

In [ ]:
# Cell 7 — Quick SFT sanity check
# Verify the fine-tuned model produces valid JSON actions

FastLanguageModel.for_inference(model)

test_input = """Task: Book a Sleeper ticket from Delhi to Mumbai

Accessibility Tree:
[ref=mn_1 role=main]
  [ref=frm_1 role=form name="Search Trains"]
    [ref=inp_1 role=textbox name="From" value=""]
    [ref=inp_2 role=textbox name="To" value=""]
    [ref=cmb_1 role=combobox name="Class" value="-- Select --"]
    [ref=btn_1 role=button name="Search"]"""

messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": test_input},
]
prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.1)
response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("Model response:")
print(response)
try:
    parsed = json.loads(response.strip())
    print(f"\nValid JSON: {parsed}")
except json.JSONDecodeError:
    print("\nWARNING: Response is not valid JSON — may need more SFT epochs")

In [ ]:
# Cell 8 — Connect to live environment for RL
# Guide §13: Deploy environment early, connect for training

import os
INFINITE_DOM_URL = os.environ.get('INFINITE_DOM_URL', 'http://localhost:8000')

try:
    from openenv.core.env_client import EnvClient
except ImportError:
    from openenv_core.env_client import EnvClient

env_client = EnvClient(INFINITE_DOM_URL)
print(f"Connected to: {INFINITE_DOM_URL}")
print(f"Metadata: {env_client.metadata()}")

In [ ]:
# Cell 9 — Baseline evaluation: random vs oracle vs SFT model
# Guide §19: Show baseline → trained improvement

import random as stdlib_random
import httpx

SCORE_MIN, SCORE_MAX = 0.01, 0.99

def run_eval_episode(policy_fn, task_id=1, seed=42, max_steps=25):
    """Run one episode with a policy function, return (completed_nodes, total_reward)."""
    r = httpx.post(f"{INFINITE_DOM_URL}/reset",
                   json={"task_id": task_id, "seed": seed}, timeout=60)
    obs = r.json()
    total_reward = 0.0
    for step in range(max_steps):
        if obs.get("done", False):
            break
        action = policy_fn(obs)
        r = httpx.post(f"{INFINITE_DOM_URL}/step",
                       json={"action": action}, timeout=30)
        obs = r.json()
        total_reward += obs.get("reward", 0) or 0
    return len(obs.get("task_progress", [])), total_reward

def random_policy(obs):
    return {"action_type": stdlib_random.choice(["click", "type", "wait"]),
            "element_ref": "btn_1", "text_value": "test", "scroll_delta": 0}

# Run baselines
print("=== Baseline: Random Policy ===")
for seed in range(5):
    nodes, reward = run_eval_episode(random_policy, task_id=1, seed=seed)
    print(f"  seed={seed}: completed={nodes}/5 reward={reward:.3f}")

print("\n=== Baseline results collected ===")
print("(Oracle eval requires local env — run separately with test_oracle.py)")

In [ ]:
# Cell 10 — GRPO RL Training (Guide §11: GRPO for verifiable tasks)
# Guide §6: Curriculum — start easy, increase difficulty
# Guide §8: Monitor for reward hacking

from trl import GRPOTrainer, GRPOConfig

# Put model back in training mode
FastLanguageModel.for_training(model)

def reward_fn(completions, prompts):
    """Score completions by sending actions to the environment."""
    rewards = []
    for completion in completions:
        try:
            action = json.loads(completion.strip())
            r = httpx.post(f"{INFINITE_DOM_URL}/step",
                           json={"action": action}, timeout=15)
            raw_reward = r.json().get("reward", 0) or 0
            # Normalize to (0, 1)
            norm = SCORE_MIN + (raw_reward - (-6.0)) / (1.5 - (-6.0)) * (SCORE_MAX - SCORE_MIN)
            rewards.append(max(SCORE_MIN, min(SCORE_MAX, norm)))
        except Exception:
            rewards.append(SCORE_MIN)
    return rewards

grpo_config = GRPOConfig(
    output_dir="./grpo_output",
    num_train_epochs=1,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    learning_rate=1e-5,
    logging_steps=5,
    save_steps=50,
    max_completion_length=200,
    num_generations=4,  # Number of rollouts per prompt
    report_to="none",
)

# Curriculum: Train on progressively harder tasks
reward_history = []
for task_id in [1, 2, 3, 4]:
    print(f"\n{'='*40}")
    print(f"GRPO Training — Task {task_id}")
    print(f"{'='*40}")
    
    # Reset env to this task
    httpx.post(f"{INFINITE_DOM_URL}/reset",
               json={"task_id": task_id, "seed": 0}, timeout=60)
    
    # Build prompts from oracle data for this task
    task_records = [r for r in records if r['task_id'] == task_id][:50]
    prompts = []
    for r in task_records:
        msgs = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": f"Task: {r['instruction']}\n\nAccessibility Tree:\n{r['observation'][:2000]}"},
        ]
        prompts.append(tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
    
    if not prompts:
        print(f"  No data for task {task_id}, skipping")
        continue
    
    prompt_dataset = Dataset.from_dict({"prompt": prompts})
    
    grpo_trainer = GRPOTrainer(
        model=model,
        tokenizer=tokenizer,
        reward_funcs=[reward_fn],
        train_dataset=prompt_dataset,
        args=grpo_config,
    )
    
    result = grpo_trainer.train()
    print(f"  Task {task_id} training loss: {result.training_loss:.4f}")
    reward_history.append({"task_id": task_id, "loss": result.training_loss})

print("\nGRPO curriculum training complete!")

In [ ]:
# Cell 11 — Save final model (Guide §16)
# Save LoRA adapters — do NOT upcast 4-bit naively

import os

# Save adapters locally
model.save_pretrained("./final_lora_adapters")
tokenizer.save_pretrained("./final_lora_adapters")
print("Saved final LoRA adapters to ./final_lora_adapters")

# Push to Hub
HF_TOKEN = os.environ.get("HF_TOKEN")
if HF_TOKEN:
    model.push_to_hub("YOUR_USERNAME/infinite-dom-agent", token=HF_TOKEN)
    tokenizer.push_to_hub("YOUR_USERNAME/infinite-dom-agent", token=HF_TOKEN)
    print("Pushed to HuggingFace Hub")
else:
    print("Set HF_TOKEN to push to Hub")

In [ ]:
# Cell 12 — Training plots (Guide §19: show improvement evidence)
# Save as PNG and embed in README

import matplotlib.pyplot as plt

# Plot 1: SFT training loss
if hasattr(trainer, 'state') and trainer.state.log_history:
    losses = [h['loss'] for h in trainer.state.log_history if 'loss' in h]
    steps = list(range(len(losses)))
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    axes[0].plot(steps, losses, 'b-', linewidth=1.5)
    axes[0].set_xlabel('Training Step')
    axes[0].set_ylabel('Loss')
    axes[0].set_title('SFT Training Loss')
    axes[0].grid(True, alpha=0.3)
    
    # Plot 2: GRPO reward per task
    if reward_history:
        task_ids = [h['task_id'] for h in reward_history]
        task_losses = [h['loss'] for h in reward_history]
        axes[1].bar(task_ids, task_losses, color=['#22c55e', '#eab308', '#f97316', '#ef4444'])
        axes[1].set_xlabel('Task ID')
        axes[1].set_ylabel('Training Loss')
        axes[1].set_title('GRPO Loss by Task (Curriculum)')
        axes[1].set_xticks([1, 2, 3, 4])
        axes[1].set_xticklabels(['Clean\nForm', 'Label\nDrift', 'Structural\nDrift', 'Full\nChaos'])
    
    plt.tight_layout()
    plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
    plt.show()
    print("Saved training_curves.png — include this in README")
else:
    print("No training history available — run SFT first")

In [ ]:
# Cell 13 — Final evaluation: trained model vs baseline
# Guide §19: Show before/after improvement

FastLanguageModel.for_inference(model)

def model_policy(obs):
    """Use the fine-tuned model to select actions."""
    tree = obs.get('a11y_tree', '')[:2000]
    instruction = obs.get('task_instruction', '')
    progress = obs.get('task_progress', [])
    
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Task: {instruction}\n\nAccessibility Tree:\n{tree}\n\nProgress: {progress}"},
    ]
    prompt = tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.1)
    response = tokenizer.decode(outputs[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    
    try:
        return json.loads(response.strip())
    except json.JSONDecodeError:
        return {"action_type": "wait", "element_ref": "", "text_value": "", "scroll_delta": 0}

print("=== Final Evaluation: Trained Model ===")
trained_results = []
for task_id in [1, 2, 3, 4]:
    nodes_list = []
    for seed in range(5):
        nodes, reward = run_eval_episode(model_policy, task_id=task_id, seed=seed)
        nodes_list.append(nodes)
    avg = sum(nodes_list) / len(nodes_list)
    trained_results.append(avg)
    print(f"  Task {task_id}: avg nodes completed = {avg:.1f}/5")

print("\n=== Comparison ===")
print(f"Random baseline: ~0-1 nodes completed per episode")
print(f"Trained model:   {sum(trained_results)/len(trained_results):.1f} nodes avg")
print("\nInclude these numbers in your README and pitch.")